# ASR Transcription with Paraformer-zh

Load WAV files from a folder, run Paraformer-zh (via FunASR) with FSMN-VAD,
and save word-level ASR results to JSONL.

In [1]:
# %pip install funasr modelscope soundfile

In [1]:
from pathlib import Path

# -- Configure these paths --
WAV_DIR = Path("../data/wav")       # folder containing .wav files
OUTPUT_PATH = Path("../data/asr_results.jsonl")

WAV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [3]:
from funasr import AutoModel

model = AutoModel(
    model="paraformer-zh",   # ASR — best Chinese accuracy at 220M params
    vad_model="fsmn-vad",    # VAD with timestamps
    punc_model="ct-punc",    # punctuation restoration
)
print("Model loaded.")

funasr version: 1.3.1.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.1


Model loaded.


In [5]:
wav_files = sorted(WAV_DIR.glob("*.wav"))
print(f"Found {len(wav_files)} WAV files in {WAV_DIR}")
for f in wav_files[:5]:
    print(f"  {f.name}")
if len(wav_files) > 5:
    print(f"  ... and {len(wav_files) - 5} more")

Found 2 WAV files in ../data/wav
  R8001_M8004_MS801.wav
  R8003_M8001_MS801.wav


In [ ]:
import json

for i, wav_path in enumerate(wav_files):
    res = model.generate(
        input=str(wav_path),
        return_raw_text=True,
        sentence_timestamp=True,
    )

    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        for item in res:
            record = {
                "file": wav_path.name,
                "text": item.get("text", ""),
                "sentences": item.get("sentence_info", []),
                "timestamp": item.get("timestamp", []),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    if (i + 1) % 10 == 0 or (i + 1) == len(wav_files):
        print(f"[{i + 1}/{len(wav_files)}] {wav_path.name}")

print(f"\nDone. Results saved to {OUTPUT_PATH}")

100%|█████████████████████████| 1/1 [00:00<00:00,  1.81it/s]
{'load_data': '0.000', 'extract_feat': '0.001', 'forward': '
rtf_avg: 1.319: 100%|█████████| 1/1 [00:00<00:00,  1.80it/s]

100%|█████████████████████████| 1/1 [00:00<00:00,  2.19it/s]
{'load_data': '0.000', 'extract_feat': '0.002', 'forward': '
rtf_avg: 0.846: 100%|█████████| 1/1 [00:00<00:00,  2.18it/s]

100%|█████████████████████████| 1/1 [00:00<00:00,  2.22it/s]
{'load_data': '0.000', 'extract_feat': '0.002', 'forward': '
rtf_avg: 0.835: 100%|█████████| 1/1 [00:00<00:00,  2.21it/s]

100%|█████████████████████████| 1/1 [00:00<00:00,  2.22it/s]
{'load_data': '0.000', 'extract_feat': '0.002', 'forward': '
rtf_avg: 0.682: 100%|█████████| 1/1 [00:00<00:00,  2.21it/s]

100%|█████████████████████████| 1/1 [00:00<00:00,  2.16it/s]
{'load_data': '0.000', 'extract_feat': '0.002', 'forward': '
rtf_avg: 0.703: 100%|█████████| 1/1 [00:00<00:00,  2.15it/s]

100%|█████████████████████████| 1/1 [00:00<00:00,  2.24it/s]
{'load_data': '0.00

In [ ]:
# Preview results
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        print(f"--- {r['file']} ---")
        print(f"  text: {r['text']}")
        if r["sentences"]:
            print(f"  sentences: {r['sentences'][:3]}")
        print()